In [1]:
import pandas as pd
import numpy as np
import ipyparallel as ipp

In [2]:
def ranked_probability_scores(observations, predictions):
    
    N = len(observations)
    rps = np.zeros(N)
    
    for i in range(N):
        
        observation_prob = np.array([0,0,0])
        observation_prob[observations[i]] = 1
            
        rps[i] = 1/2*( (np.cumsum(predictions[i])-np.cumsum(observation_prob))**2 ).sum()

    return rps

In [3]:
def validation_step(df_X, df_y, winter, start_winter, max_tree_depth, min_leaf_size, split_crit, n_trees, chunksize=1):
    
    is_validation_data = (winter >= start_winter) & (winter < start_winter + chunksize)
    X_valid = df_X[is_validation_data].values
    X_train = df_X[~is_validation_data].values
    y_valid = df_y[is_validation_data].values[:,0]
    y_train = df_y[~is_validation_data].values[:,0]
        
    classifier = RandomForestClassifier(n_estimators=n_trees, criterion=split_crit, max_depth=max_tree_depth, 
                                        min_samples_leaf=min_leaf_size, random_state=42)
    
    classifier.fit(X_train, y_train)
    y_pred = classifier.predict_proba(X_valid)
    rps = ranked_probability_scores(y_valid, y_pred)
    
    return rps.mean()

In [4]:
def cross_validation_k_fold(leave_out_feature, df_features, df_y, max_tree_depth, min_leaf_size, split_crit, n_trees, k=60, chunksize=1):

    df_X = df_features.drop(columns=leave_out_feature)

    year = df_X.index.year
    month = df_X.index.month
    winter = year.where(month < 12, year+1)
    start_winters = np.arange(winter[0], winter[-1]+1, chunksize)

    rps_k_fold = np.zeros(k)
    
    for i in range(k):
        rps_k_fold[i] = validation_step(df_X, df_y, winter, start_winters[i], max_tree_depth, min_leaf_size, split_crit, n_trees, chunksize)
        
    return rps_k_fold.mean()

In [5]:
df_features = pd.read_csv('features/features_Kiefer_etal_week3.csv')
df_target = pd.read_csv('targets/northern_temperature_terc_cat_new.csv')

In [6]:
df_features['time'] = pd.to_datetime(df_features['time'])
df_features = df_features.set_index('time')

df_target['time'] = pd.to_datetime(df_target['time'])
df_target = df_target.set_index('time')

In [7]:
df_target_train = df_target[df_target.index < "2010-07"]

In [8]:
drop = []

In [9]:
df_features_sel = df_features.drop(columns=drop)
df_features_train = df_features_sel[df_features_sel.index < "2010-07"]
len(df_features_train.columns)

37

In [10]:
n = 37
cluster = ipp.Cluster(n = n)
c = cluster.start_and_connect_sync()

Starting 37 engines with <class 'ipyparallel.cluster.launcher.LocalEngineSetLauncher'>
100%|██████████| 37/37 [00:05<00:00,  6.24engine/s]


In [11]:
dview = c[:]
dview.block = True 

In [12]:
dview.execute('from sklearn.ensemble import RandomForestClassifier')
dview.execute('import numpy as np')

<AsyncResult(execute): finished>

In [13]:
N = 1000
d_max = None
min_samples_leaf = 100
criterion = 'entropy'

In [14]:
mydict = dict(df_features_train=df_features_train, df_target_train=df_target_train, cross_validation_k_fold=cross_validation_k_fold, validation_step=validation_step, 
              ranked_probability_scores=ranked_probability_scores, N=N, d_max=d_max, min_samples_leaf=min_samples_leaf, criterion=criterion)
dview.push(mydict)

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [15]:
lview = c.load_balanced_view()
lview.block = True

In [16]:
%%time
def wrapper(i):
    return(cross_validation_k_fold(df_features_train.columns[i], df_features_train, df_target_train, d_max, min_samples_leaf, criterion, N))

result_list = lview.map(wrapper, range(n))

CPU times: user 7.15 s, sys: 3.04 s, total: 10.2 s
Wall time: 23min 41s


In [17]:
np.array(result_list)[np.array(result_list).argsort()]

array([0.21321204, 0.21321404, 0.21329689, 0.21329808, 0.21330102,
       0.2133195 , 0.21334478, 0.2133522 , 0.2133669 , 0.21337989,
       0.2133929 , 0.21341345, 0.21344896, 0.21345164, 0.21347415,
       0.21348899, 0.21349884, 0.21350341, 0.21351037, 0.21351096,
       0.21351746, 0.21351931, 0.21352579, 0.21358872, 0.21359636,
       0.21360086, 0.21361785, 0.21361795, 0.2136292 , 0.21367413,
       0.21374706, 0.21379338, 0.21379362, 0.21379961, 0.21415807,
       0.2149364 , 0.2153393 ])

In [18]:
df_features_train.columns[np.array(result_list).argsort()]

Index(['q850_std', 'z300_max', 'z100_max', 'z300_std', 'z500_std', 'u300_mean',
       'z300_mean', 'slp_std', 'u10_max', 'u10_mean', 'q850_mean', 'z850_min',
       't850_max', 'z100_mean', 'u300_min', 'slp_min', 't850_min', 'z500_max',
       'q850_max', 'q850_min', 'z300_min', 'slp_max', 'u300_max', 't850_mean',
       'slp_mean', 'z500_min', 'z850_std', 'u300_std', 'z850_max', 'month',
       'u10_std', 'z100_std', 'z850_mean', 'z500_mean', 'u10_min', 'z100_min',
       't850_std'],
      dtype='str')

In [19]:
features_ordered = df_features_train.columns[np.array(result_list).argsort()]

In [20]:
mydict = dict(df_features_train=df_features_train, df_target_train=df_target_train, cross_validation_k_fold=cross_validation_k_fold, validation_step=validation_step, 
              ranked_probability_scores=ranked_probability_scores, N=N, d_max=d_max, min_samples_leaf=min_samples_leaf, criterion=criterion, features_ordered=features_ordered)
dview.push(mydict)

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [21]:
%%time
def wrapper(i):
    return(cross_validation_k_fold(features_ordered[:i], df_features_train, df_target_train, d_max, min_samples_leaf, criterion, N))

result_list = lview.map(wrapper, range(n))

CPU times: user 5.62 s, sys: 2.56 s, total: 8.18 s
Wall time: 21min 21s


In [22]:
np.array(result_list)

array([0.21358843, 0.21321204, 0.21327514, 0.21303017, 0.21306943,
       0.21277787, 0.21248133, 0.21270307, 0.21224595, 0.21189517,
       0.21163499, 0.21144415, 0.21125746, 0.21099615, 0.21112649,
       0.21098584, 0.21102886, 0.21073293, 0.21049958, 0.21054507,
       0.21013062, 0.21007563, 0.21017206, 0.21005159, 0.20973257,
       0.20959989, 0.20959303, 0.20898029, 0.2090111 , 0.20927932,
       0.20875406, 0.20918346, 0.21024675, 0.21108734, 0.21536264,
       0.2163603 , 0.22410721])

In [23]:
features_ordered[:np.array(result_list).argmin()]

Index(['q850_std', 'z300_max', 'z100_max', 'z300_std', 'z500_std', 'u300_mean',
       'z300_mean', 'slp_std', 'u10_max', 'u10_mean', 'q850_mean', 'z850_min',
       't850_max', 'z100_mean', 'u300_min', 'slp_min', 't850_min', 'z500_max',
       'q850_max', 'q850_min', 'z300_min', 'slp_max', 'u300_max', 't850_mean',
       'slp_mean', 'z500_min', 'z850_std', 'u300_std', 'z850_max', 'month'],
      dtype='str')